In [1]:
import hdc
hdc.connect()

Exception reporting mode: Minimal
Version: hdc 0.6.7, duckdb 1.4.3 (memory=29.8 GiB, threads=15, size=1.969 GB)
Province Code: 49
Last Update: 2026-06-19


In [2]:
prov_c = '49' ;
start_d = '2025-10-01' ; 
end_d = '2026-09-30' ; 

# start_d1 = '2024-10-01' ; 
# end_d1 = '2025-09-30' ; 
# start_d2 = '2025-10-01' ; 
# end_d2 = '2026-09-30'

In [3]:
%%sql
/* -- Telemedicine รายหน่วยบริการ-- */
DROP TABLE IF EXISTS telemed_hosp ;
CREATE TABLE telemed_hosp
AS
SELECT c.amp_code,c.amp_name
    ,c.hoscode AS hospcode,c.hosname AS hospname
    ,c1.hostypecode,c1.hostypename
    ,c.mcode,c.m_name,c.dep_name
    ,x.Service69,x.Telemed69
    ,round((x.Telemed69 / x.Service69 * 100),2) AS PercentTelemed69
FROM (
    SELECT s.hospcode
        ,count(DISTINCT CONCAT(s.hospcode,'-',s.pid,'-',s.seq)) AS Service69
        ,count(DISTINCT IF(s.typein = '5',CONCAT(s.hospcode,'-',s.pid,'-',s.seq),NULL)) AS Telemed69      
      
    FROM service s
        INNER JOIN person p ON s.hospcode=p.hospcode AND s.pid=p.pid
    WHERE s.date_serv BETWEEN '{{start_d}}' AND '{{end_d}}'
        AND s.typein IN('2','3','5')
    GROUP BY s.hospcode
    ) AS x 
    INNER JOIN chospital c ON c.hoscode=x.hospcode 
    LEFT JOIN chostype c1 ON c1.hostypecode=c.hostype 
ORDER BY  c.amp_code,c.hoscode  ;


Count
44


In [4]:
from datetime import datetime
# ชื่อจาก วันที่
today = datetime.today().strftime('%Y%m%d')
filename = f'./Export/{today}_{prov_c}_telemed_hosp_typein235.xlsx'

In [5]:
%%sql
/* Export to excel แบบนี้ work สุด */
install excel;
load excel;
COPY (SELECT * FROM  telemed_hosp ORDER BY amp_code,hospcode)
TO '{{filename}}' (FORMAT xlsx, HEADER true);

Count
44


In [6]:
hdc.close()